# 76 - Five-critic latent Q-guidance, worker 3/4

This worker runs shard 3 of the frozen four-shard PRO160 experiment: 40 held-out position-perturbation identities times five latent-guidance critics = 200 new rollouts. Every arm uses the ordinary 10-step PI0.5 decoder, one normalized Q-ascent update at zero-based Euler step 3 with RMS 0.005, executes the first 10 of 50 actions, and replans.

Periodic tables print every 10 identities completed under all five arms and include the exact matched historical stock-VLA outcomes from notebook 68. Videos, frames, and generated chunks are off. This worker resolves all five checkpoints directly from the mounted Drive.

## 1. Setup

In [ ]:
EXTRAS = 'sim'
SETUP_ENV = True
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

## 2. Resolve the five checkpoints from Drive

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

SHARD_COUNT = 4
SHARD_INDEX = 3
EPISODE_LIMIT = None  # set to 1 only for a five-rollout smoke test
LATENT_UPDATE_RMS = 0.005
DRIVE_ROOT = Path('/content/drive/MyDrive')

def exactly_one(pattern):
    matches = sorted(DRIVE_ROOT.glob(pattern))
    if len(matches) != 1:
        raise ValueError(
            f'Expected exactly one match for {pattern}; found '
            f'{len(matches)}: {[str(path) for path in matches]}')
    return matches[0]

ORIGINAL_Q50_CHECKPOINT_PATH = exactly_one(
    'pnp_qplanning_corrector/pcpcds-*/q50_full/checkpoint_step_008000.pt')
FAILURE_CHECKPOINT_PATH = exactly_one(
    'pnp_qplanning_priority/pcpcds-*/q50_priority_failure_full/'
    'checkpoint_step_006000.pt')
EPISODE_U20_CHECKPOINT_PATH = exactly_one(
    'pnp_qplanning_priority/pcpcds-*/q50_priority_episode_u20_full/'
    'checkpoint_step_006000.pt')
U20_4CHUNK_CHECKPOINT_PATH = exactly_one(
    'pnp_qplanning_priority/pcpcds-*/q50_priority_u20_4chunk_full/'
    'checkpoint_step_006000.pt')
U20_8CHUNK_CHECKPOINT_PATH = exactly_one(
    'pnp_qplanning_priority/pcpcds-*/q50_priority_u20_8chunk_full/'
    'checkpoint_step_006000.pt')

print({
    'worker': f'{SHARD_INDEX}/{SHARD_COUNT}',
    'identities': 40 if EPISODE_LIMIT is None else EPISODE_LIMIT,
    'new_rollouts': 200 if EPISODE_LIMIT is None else 5 * EPISODE_LIMIT,
    'checkpoints': {
        'original': str(ORIGINAL_Q50_CHECKPOINT_PATH),
        'failure': str(FAILURE_CHECKPOINT_PATH),
        'episode_u20': str(EPISODE_U20_CHECKPOINT_PATH),
        'u20_4chunk': str(U20_4CHUNK_CHECKPOINT_PATH),
        'u20_8chunk': str(U20_8CHUNK_CHECKPOINT_PATH),
    },
    'periodic_print_every_complete_identities': 10,
    'historical_stock': 'exact matched notebook-68 outcomes; not rerun',
})

## 3. Run this shard's five latent-guidance arms

In [ ]:
from pnp.qplanning_gradient_eval_experiment import (
    QGUIDE_FIVE_EVAL_EXPERIMENT,
    run_qplanning_latent_guidance_heldout160,
)

report = run_qplanning_latent_guidance_heldout160(
    original_checkpoint_path=ORIGINAL_Q50_CHECKPOINT_PATH,
    failure_checkpoint_path=FAILURE_CHECKPOINT_PATH,
    episode_u20_checkpoint_path=EPISODE_U20_CHECKPOINT_PATH,
    u20_4chunk_checkpoint_path=U20_4CHUNK_CHECKPOINT_PATH,
    u20_8chunk_checkpoint_path=U20_8CHUNK_CHECKPOINT_PATH,
    shard_count=SHARD_COUNT,
    shard_index=SHARD_INDEX,
    episode_limit=EPISODE_LIMIT,
    update_rms=LATENT_UPDATE_RMS,
    experiment=QGUIDE_FIVE_EVAL_EXPERIMENT,
)
report

## 4. Audit persisted guidance settings and telemetry

In [ ]:
from pnp.qplanning_gradient_eval_experiment import (
    validate_qplanning_latent_guidance_sentinel,
)

validate_qplanning_latent_guidance_sentinel(
    checkpoint_ids=report['checkpoint_ids'],
    experiment=report['experiment'],
)